In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor

In [2]:
pd_train=pd.read_csv("train.csv")

In [3]:
pd_train.head()

,ID,Gender,Age,Occupation,Physical_Activity_Level,Stress_Level,BMI_Category,Heart_Rate,Daily_Steps,Sleep_Disorder,Temp,Mood,History,Quality_of_Sleep,Sleep_Duration
0,311,Female,49,Nurse,90,8,Overweight,75,10000,Sleep Apnea,-1.278341,0.440786,5.407088,7.474116,6.2
1,207,Male,30,Doctor,75,6,Normal,70,8000,NaN,-0.826910,0.691399,5.839617,7.979600,7.7
2,169,Male,35,Engineer,60,4,Normal,65,5000,NaN,1.139555,0.909316,7.039570,5.902686,7.3
3,149,Male,39,Lawyer,60,5,Normal,68,8000,NaN,2.069801,0.741677,4.464130,8.127826,7.2
4,98,Female,50,Nurse,90,8,Overweight,75,10000,Sleep Apnea,-2.907425,0.418233,3.068291,8.099139,6.0


In [4]:
pd_test=pd.read_csv("test.csv")

In [5]:
pd_test.head()

,ID,Gender,Age,Occupation,Physical_Activity_Level,Stress_Level,BMI_Category,Heart_Rate,Daily_Steps,Sleep_Disorder,Temp,Mood,History,Quality_of_Sleep
0,3,Male,31,Doctor,30,8,Normal,72,5000,NaN,-1.627562,0.474605,6.029331,5.970977
1,234,Male,43,Salesperson,45,7,Overweight,72,6000,Sleep Apnea,3.311945,0.567641,5.366240,5.824984
2,187,Female,44,Teacher,45,4,Overweight,65,6000,Insomnia,0.800264,0.924036,6.645139,7.465405
3,220,Male,33,Doctor,30,8,Normal,72,5000,NaN,4.397880,0.512827,8.466199,7.946357
4,106,Male,30,Doctor,75,6,Normal,70,8000,NaN,-0.383691,0.634333,6.751696,6.343607


In [6]:
feature_cols = [col for col in pd_train.columns if col not in ['ID', 'Sleep_Duration']]

cat_cols = pd_train[feature_cols].select_dtypes(include=['object', 'string']).columns.tolist()
num_cols = pd_train[feature_cols].select_dtypes(include=['int64', 'float64']).columns.tolist()

print("categorical column:", cat_cols)
print("numerical column:", num_cols)

categorical column: ['Gender', 'Occupation', 'BMI_Category', 'Sleep_Disorder']
numerical column: ['Age', 'Physical_Activity_Level', 'Stress_Level', 'Heart_Rate', 'Daily_Steps', 'Temp', 'Mood', 'History', 'Quality_of_Sleep']


In [7]:
pd_train['Sleep_Disorder'] = pd_train['Sleep_Disorder'].fillna('None')
pd_test['Sleep_Disorder'] = pd_test['Sleep_Disorder'].fillna('None')
for col in cat_cols:
    le = LabelEncoder()
    full_data = list(pd_train[col].astype(str)) + list(pd_test[col].astype(str))
    le.fit(full_data)
    
    pd_train[col] = le.transform(pd_train[col].astype(str))
    pd_test[col] = le.transform(pd_test[col].astype(str))

X = pd_train[feature_cols]
y = pd_train['Sleep_Duration']
X_test = pd_test[feature_cols]


In [8]:
pd_train.head()

,ID,Gender,Age,Occupation,Physical_Activity_Level,Stress_Level,BMI_Category,Heart_Rate,Daily_Steps,Sleep_Disorder,Temp,Mood,History,Quality_of_Sleep,Sleep_Duration
0,311,0,49,5,90,8,3,75,10000,2,-1.278341,0.440786,5.407088,7.474116,6.2
1,207,1,30,1,75,6,0,70,8000,1,-0.826910,0.691399,5.839617,7.979600,7.7
2,169,1,35,2,60,4,0,65,5000,1,1.139555,0.909316,7.039570,5.902686,7.3
3,149,1,39,3,60,5,0,68,8000,1,2.069801,0.741677,4.464130,8.127826,7.2
4,98,0,50,5,90,8,3,75,10000,2,-2.907425,0.418233,3.068291,8.099139,6.0


In [9]:
pd_test.head()

,ID,Gender,Age,Occupation,Physical_Activity_Level,Stress_Level,BMI_Category,Heart_Rate,Daily_Steps,Sleep_Disorder,Temp,Mood,History,Quality_of_Sleep
0,3,1,31,1,30,8,0,72,5000,1,-1.627562,0.474605,6.029331,5.970977
1,234,1,43,7,45,7,3,72,6000,2,3.311945,0.567641,5.366240,5.824984
2,187,0,44,10,45,4,3,65,6000,0,0.800264,0.924036,6.645139,7.465405
3,220,1,33,1,30,8,0,72,5000,1,4.397880,0.512827,8.466199,7.946357
4,106,1,30,1,75,6,0,70,8000,1,-0.383691,0.634333,6.751696,6.343607


In [10]:
kf = KFold(n_splits=5, shuffle=True, random_state=0)
oof_predictions = np.zeros(len(X))
test_predictions = np.zeros(len(X_test))

mse_scores = []


for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
# or HistGradientBoostingRegressor()
    model = RandomForestRegressor(
    n_estimators=100, 
    random_state=42, 
    n_jobs=-1
    )

    model.fit(X_train, y_train)
    model.fit(X_train, y_train)

    val_preds = model.predict(X_val)
    oof_predictions[val_idx] = val_preds
    
    fold_mse = mean_squared_error(y_val, val_preds)
    mse_scores.append(fold_mse)
    print(f"Fold {fold + 1} MSE score: {fold_mse:.4f}")
    
    test_predictions += model.predict(X_test) / kf.n_splits


overall_mse = mean_squared_error(y, oof_predictions)
print(f"\nTotal OOF crossing validation MSE score: {overall_mse:.4f}")

Fold 1 MSE score: 0.0448
Fold 2 MSE score: 0.0198
Fold 3 MSE score: 0.0144
Fold 4 MSE score: 0.0143
Fold 5 MSE score: 0.0163

Total OOF crossing validation MSE score: 0.0219


In [11]:
submission = pd_test[['ID']].copy()
submission['Sleep_Duration'] = test_predictions

submission.to_csv('submission_RF.csv', index=False)
print(submission.head())

    ID  Sleep_Duration
0    3          6.1154
1  234          6.4350
2  187          6.5968
3  220          6.1168
4  106          7.7140
